# Notebook 02 — Performance & Parallelism

Python loops are slow. This notebook shows the three-step ladder for speeding up
numerical code:

1. **Vectorise** with NumPy (almost always the right first step)
2. **Compile** with Numba for loops that cannot be vectorised
3. **Parallelise** with `ProcessPoolExecutor` for independent tasks

See also `examples/coding/performance/comparison.py` for a standalone script
covering a physics-motivated sequential simulation.

In [ ]:
import random
import time
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


def _find_style() -> Path | None:
    path = Path.cwd()
    for _ in range(6):
        candidate = path / "examples" / "plotting" / "styles" / "nestling.mplstyle"
        if candidate.exists():
            return candidate
        path = path.parent
    return None


_style = _find_style()
if _style:
    plt.style.use(str(_style))

NORM  = 1.66e-18
E_REF = 1e5
GAMMA = 2.37

## Task A — 1-D vectorisation

The classic comparison: evaluate the IceCube flux at 1 million energies.
NumPy moves the loop from the Python interpreter into compiled C.

In [ ]:
E_1D = np.logspace(3, 7, 1_000_000)

# Python loop
t0 = time.perf_counter()
result_loop = [NORM * (e / E_REF) ** (-GAMMA) * e**2 for e in E_1D]
t_loop = time.perf_counter() - t0
print(f"Python loop       : {t_loop * 1e3:7.1f} ms")

# NumPy vectorised
t0 = time.perf_counter()
result_vec = NORM * (E_1D / E_REF) ** (-GAMMA) * E_1D**2
t_vec = time.perf_counter() - t0
print(f"NumPy vectorised  : {t_vec * 1e3:7.1f} ms   speedup = {t_loop / t_vec:.0f}×")

assert np.allclose(result_loop, result_vec)

## Task B — Numba JIT: Monte Carlo Pi estimation

Estimate π by generating N random points in the unit square and counting
those that fall inside the unit circle.

This loop is **genuinely sequential** (each sample is independent, but the
result is a running count — there is no array-shaped output to vectorise into).
Numba compiles the exact same Python loop to machine code with a single decorator.

**Key observation**: the first Numba call includes JIT compilation time.
The second call reuses the compiled code — this is why you always warm up
before benchmarking Numba.

In [ ]:
N = 5_000_000   # Monte Carlo samples


def estimate_pi_python(n: int) -> float:
    """Pure Python — one interpreter step per sample."""
    count = 0
    for _ in range(n):
        x = random.random()
        y = random.random()
        if x**2 + y**2 <= 1.0:
            count += 1
    return 4.0 * count / n


def estimate_pi_numpy(n: int) -> float:
    """NumPy — generate all samples at once, then apply boolean mask."""
    x = np.random.random(n)
    y = np.random.random(n)
    return 4.0 * np.sum(x**2 + y**2 <= 1.0) / n

In [ ]:
try:
    from numba import njit

    @njit(cache=True)
    def estimate_pi_numba(n: int) -> float:
        """
        Numba JIT — same code as the Python version, single decorator added.

        Numba reimplements random.random() with its own internal RNG.
        The state is independent of Python's random module.
        """
        count = 0
        for _ in range(n):
            x = random.random()
            y = random.random()
            if x**2 + y**2 <= 1.0:
                count += 1
        return 4.0 * count / n

    HAS_NUMBA = True
except ImportError:
    HAS_NUMBA = False
    print("numba not installed — run: pip install numba")

print(f"Numba available: {HAS_NUMBA}")

In [ ]:
# Python
t0 = time.perf_counter()
pi_py = estimate_pi_python(N)
t_py = time.perf_counter() - t0
print(f"Python            : {t_py * 1e3:7.0f} ms   π ≈ {pi_py:.5f}")

# NumPy
t0 = time.perf_counter()
pi_np = estimate_pi_numpy(N)
t_np = time.perf_counter() - t0
print(f"NumPy (vectorised): {t_np * 1e3:7.0f} ms   π ≈ {pi_np:.5f}   speedup = {t_py / t_np:.0f}×")

if HAS_NUMBA:
    # First Numba call — includes JIT compilation
    t0 = time.perf_counter()
    pi_nb1 = estimate_pi_numba(N)
    t_nb1 = time.perf_counter() - t0
    print(f"Numba (1st call)  : {t_nb1 * 1e3:7.0f} ms   π ≈ {pi_nb1:.5f}   ← includes JIT compilation")

    # Second Numba call — compiled machine code only
    t0 = time.perf_counter()
    pi_nb2 = estimate_pi_numba(N)
    t_nb2 = time.perf_counter() - t0
    print(f"Numba (2nd call)  : {t_nb2 * 1e3:7.0f} ms   π ≈ {pi_nb2:.5f}   speedup = {t_py / t_nb2:.0f}×")

    print(f"\nCompilation overhead: {(t_nb1 - t_nb2) * 1e3:.0f} ms — paid once per session")

In [ ]:
if HAS_NUMBA:
    labels   = ["Python", "NumPy", "Numba\n(compiled)"]
    times_ms = [t_py * 1e3, t_np * 1e3, t_nb2 * 1e3]

    fig, ax = plt.subplots(figsize=(4, 3))
    bars = ax.bar(labels, times_ms)
    ax.set_ylabel("Time [ms]")
    ax.set_title(f"Monte Carlo π, N = {N:,}")
    for bar, val in zip(bars, times_ms):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.02,
                f"{val:.0f} ms", ha="center", va="bottom", fontsize=7)
    ax.set_yscale("log")
    fig.tight_layout()
    plt.show()

## Task C — Parallel independent simulations

`ProcessPoolExecutor` distributes independent tasks across CPU cores.
Each worker is a separate process, so Python's GIL does not block parallelism.

> The worker function must be **self-contained** — it runs in a subprocess that
> does not have access to Jupyter notebook cell state. All constants it needs
> must be defined inside the function or at module level.

In [ ]:
def simulate_one(seed: int) -> float:
    """
    Simulate one toy neutrino telescope observation.

    Samples energies uniformly in log-space and applies Gaussian
    energy resolution (10%).  Returns the mean reconstructed energy.

    Self-contained so it can be pickled by ProcessPoolExecutor.
    """
    import numpy as _np   # explicit import — this runs in a worker subprocess
    rng = _np.random.default_rng(seed)
    log_E = rng.uniform(3.0, 7.0, size=500)         # uniform in log10(E/GeV)
    true_E = 10.0 ** log_E
    reco_E = true_E * rng.normal(1.0, 0.1, size=500)  # 10% energy resolution
    return float(reco_E.mean())


seeds = list(range(200))

# Sequential
t0 = time.perf_counter()
results_seq = [simulate_one(s) for s in seeds]
t_seq = time.perf_counter() - t0
print(f"Sequential        : {t_seq * 1e3:6.0f} ms")

# Parallel — 4 workers
t0 = time.perf_counter()
with ProcessPoolExecutor(max_workers=4) as pool:
    results_par = list(pool.map(simulate_one, seeds))
t_par = time.perf_counter() - t0
print(f"Parallel (4 cores): {t_par * 1e3:6.0f} ms   speedup ≈ {t_seq / t_par:.1f}×")

assert np.allclose(results_seq, results_par)

## Summary

| Approach | When to use |
|----------|-------------|
| NumPy vectorisation | Array operations — always try this first |
| Numba `@njit` | Loops with step-to-step dependencies, or where NumPy's vectorised rewrite is too complex |
| `ProcessPoolExecutor` | Many independent tasks (different seeds, input files) |

Also see `examples/coding/performance/comparison.py` for a physics-motivated
sequential simulation (muon energy loss) that shows all three approaches
on a task where the inner loop genuinely cannot be vectorised.

See the [Performance & Parallelism lesson](../../../docs/tracks/05-coding/lesson-06.md).